# NSOS v11 — Federated training on Google Colab

**Architecture:** Colab provides the GPU compute, Google Drive (5TB on AI Pro) stores the checkpoints, the local repo on Windows owns the authoritative history.

**What this notebook does, in order:**
1. Mount your Google Drive (read/write `MyDrive/nsos_v11/`).
2. Clone (or update) the repo from GitHub.
3. Auto-detect the Colab GPU (T4 / A100 / L4) and pick the matching profile.
4. Build `nsos_ext.so` (cached in Drive per git SHA, so only the first session rebuilds).
5. Fetch the v11 distillation bundle (cached in Drive after first build).
6. Start training with auto-checkpoint to Drive every N steps.
7. On session timeout, you re-open this notebook and step 6 resumes from the last Drive checkpoint.

**Model size:** 80M params (16 layers, d_model=640) by default — the Chinchilla-optimal sweet spot for a ~1.6B token budget.
To switch back to the 40M variant, set `os.environ['NSOS_PARAM_SCALE'] = '40m'` before running cell 3.

**Expected wall time on T4 (free or Pro):** ~24 h for the full curriculum (80M variant), split across 2-3 sessions of 8-12h each.
**Expected wall time on A100 (Pro+ or paid compute units):** ~5-6 h (80M variant), fits in one session.

**Storage budget in Drive:** ~6-15 GB per run (80M checkpoints are ~700 MB each; full history of 6 phase checkpoints + bundle).  Your 5TB has room for hundreds of runs.

## Step 1 — Mount Drive and check what's available

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, subprocess
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/nsos_v11')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Quick GPU check — confirms what Colab gave you this session
print('---  GPU allocation  ---')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], check=False)
print()
print('---  Drive contents  ---')
for entry in sorted(DRIVE_ROOT.iterdir()) if DRIVE_ROOT.exists() else []:
    print(f'  {entry.name}/')
if not list(DRIVE_ROOT.iterdir()):
    print('  (empty — first run, that is normal)')

## Step 2 — Clone (or update) the repo

Replace `<YOUR_GIT_REMOTE>` below with your actual repo URL.
If your repo is private, set up a GitHub PAT in Colab Secrets (key icon on left sidebar) as `GH_TOKEN` and the cell below will use it.

In [ ]:
REPO_URL = 'https://github.com/vitorGgC569/reimagined.git'
REPO_ROOT = Path('/content/reimagined')

try:
    from google.colab import userdata
    GH_TOKEN = userdata.get('GH_TOKEN')
except Exception:
    GH_TOKEN = None

def gh_url():
    if GH_TOKEN and 'github.com' in REPO_URL:
        return REPO_URL.replace('https://', f'https://{GH_TOKEN}@')
    return REPO_URL

if REPO_ROOT.exists():
    print(f'[git] repo exists at {REPO_ROOT} — pulling latest')
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
else:
    print(f'[git] cloning {REPO_URL} -> {REPO_ROOT}')
    subprocess.run(['git', 'clone', '--depth', '1', gh_url(), str(REPO_ROOT)], check=True)

os.environ['REPO_ROOT'] = str(REPO_ROOT)
os.environ['DRIVE_ROOT'] = str(DRIVE_ROOT)

# Show commit we are running
subprocess.run(['git', '-C', str(REPO_ROOT), 'log', '-1', '--oneline'], check=True)

## Step 3 — Bootstrap: detect GPU, build/cache .so, fetch bundle

This calls `colab/colab_bootstrap.py` which:
- Detects your GPU and picks the matching `train_curriculum.py` profile (`hybrid_v11_colab_{t4,a100,l4}`)
- Builds the `nsos_ext.so` if Drive doesn't already have one for this commit/arch (cached afterward)
- Extracts the v11 dataset bundle from Drive (if cached) or signals that the next cell needs to build it

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / 'colab'))

# ── Choose model size ───────────────────────────────────────────────────
# '80m' (default) = 16 layers, d_model=640, ~80M params, ~1.6B token target
# '40m'           = 12 layers, d_model=512, ~40M params, ~800M token target
# Override here BEFORE importing colab_bootstrap (it reads NSOS_PARAM_SCALE):
import os
os.environ.setdefault('NSOS_PARAM_SCALE', '80m')

from colab_bootstrap import setup_environment
info = setup_environment(verbose=True)

## Step 4 — Prepare the dataset bundle

For this first v11 experiment we use the **v4 distillation bundle** that already ships in the repo (the same data v10 trained on).

Why not the expanded v11 dataset (Cosmopedia/TinyStories/SmolTalk/etc) we discussed? The `fetch_real_datasets.py` script can download those, but `nsos_curriculum_lib.py` (the builder) doesn't yet know how to weave them into the phase bundles. That wiring is a follow-up task (~2h of work).

**What this means for the experiment:** v11 here is testing "**does 80M params + more training steps help on the SAME data**?" rather than "does new data help?". Both are valuable signals, just answering different questions.

If you want to test the new-data hypothesis later, the follow-up is to extend `nsos_curriculum_lib.py` to source from Cosmopedia/TinyStories — separate work.

In [ ]:
# Use the v4 bundle shipped in the repo.  It's small (2.7MB) and already
# tokenized + phase-organized.  Confirm it exists and report what's in it.
import os, sys
from pathlib import Path

V4_BUNDLE = REPO_ROOT / 'OXN/nsos/scripts/distillation_bundle_v4'

if not V4_BUNDLE.exists():
    raise FileNotFoundError(
        f"v4 bundle not found at {V4_BUNDLE}.  "
        f"Did the git clone succeed?  Re-run cell 2."
    )

# Show what we have
print(f'[data] using bundle: {V4_BUNDLE}')
print(f'[data] manifest:')
import json
manifest = json.loads((V4_BUNDLE / 'curriculum_manifest.json').read_text('utf-8'))
total_train = sum(p['train_samples'] for p in manifest['phases'])
total_eval = sum(p['eval_samples'] for p in manifest['phases'])
print(f'[data]   total train samples: {total_train}')
print(f'[data]   total eval samples:  {total_eval}')
for p in manifest['phases']:
    print(f"[data]   - {p['name']:<22} train={p['train_samples']:>4} eval={p['eval_samples']:>3}")

# Cache to Drive so future sessions can skip this (just a confirmation step really)
import shutil
cache_dir = DRIVE_ROOT / '_datasets_processed'
cache_dir.mkdir(parents=True, exist_ok=True)
cache_zip = cache_dir / 'distillation_bundle_v4.zip'
if not cache_zip.exists():
    print(f'[data] copying bundle to Drive cache (one-time, ~2.7 MB)...')
    shutil.make_archive(str(cache_zip.with_suffix('')), 'zip', V4_BUNDLE)
    print(f'[data] cached -> {cache_zip}')
else:
    print(f'[data] Drive cache already present -> {cache_zip}')

## Step 5 — Start (or resume) training

The trainer is configured to:
- Use the profile your GPU resolved to (`info['profile']`)
- Save a checkpoint every N steps DIRECTLY into Drive (no copy step)
- Resume automatically from the latest Drive checkpoint if found

If your Colab session dies in the middle, just re-open this notebook and re-run this cell — it picks up where it left off.

In [ ]:
import time
RUN_NAME = os.environ.get('NSOS_RUN_NAME',
    f"v11_{info['profile'].split('_')[-1]}_run01_{time.strftime('%Y-%m-%d')}")

RUN_DIR_DRIVE = DRIVE_ROOT / 'runs' / RUN_NAME
RUN_DIR_DRIVE.mkdir(parents=True, exist_ok=True)

print(f'[train] profile     = {info["profile"]}')
print(f'[train] run_name    = {RUN_NAME}')
print(f'[train] drive_dir   = {RUN_DIR_DRIVE}')
print(f'[train] ext_so      = {info["ext_so"]}')
print()

env = os.environ.copy()
env['PYTHONPATH'] = str(Path(info['ext_so']).parent)
env['NSOS_BUILD_DIR'] = info['build_dir']

# Hand the trainer the Drive path directly — checkpoints land in Drive, no copy step needed.
cmd = [
    sys.executable, '-u',
    str(REPO_ROOT / 'OXN/nsos/scripts/train_curriculum.py'),
    '--profile', info['profile'],
    '--run-dir', str(RUN_DIR_DRIVE),
    '--checkpoint-every', '100',
    '--resume-from-latest',  # picks up from last checkpoint if present
]
print('[train] $', ' '.join(cmd))
subprocess.run(cmd, env=env, check=False)

## Step 6 — Verify training state

Show what's in Drive so you know everything synced.  Your local `drive_sync.py` daemon will pull this down automatically (if running).

In [ ]:
print(f'[verify] contents of {RUN_DIR_DRIVE}:')
for f in sorted(RUN_DIR_DRIVE.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:<40}  {size_mb:>8.1f} MB')

print()
print('--- Next steps ---')
print(f'  Local sync: python OXN/nsos/scripts/drive_sync.py --once')
print(f'  Or daemon:  python OXN/nsos/scripts/drive_sync.py --run {RUN_NAME} --interval 300')